# M-Shots Learning


In this lab, we'll explore zero-shot, one-shot, and few-shot prompting. We'll compare how examples and explicit instructions influence the consistency and format of model responses.


## Setup

Install the dependencies if needed (`pip install -U openai python-dotenv`). Create a `.env` file containing `OPENAI_API_KEY=your_api_key`, or set the environment variable in your shell. Never commit your API key to version control.


In [4]:
import os
from dotenv import load_dotenv, find_dotenv
from openai import OpenAI

load_dotenv(find_dotenv())

if not os.getenv("OPENAI_API_KEY"):
    raise ValueError("Set OPENAI_API_KEY in your environment or .env file.")

client = OpenAI()
MODEL = "gpt-4o-mini"


## Few-shot prompting

A *shot* is an example included in a prompt:

- **Zero-shot:** no examples; the model relies on the instructions and its prior training.
- **One-shot:** one example.
- **Few-shot:** multiple examples.

Examples can help a model follow a desired style, label set, or output format. There is no universally correct number of examples: results depend on the task, the examples, and the model. Examples also consume input tokens, so use a small, representative set and evaluate the results.

This notebook uses OpenAI's **Responses API** with `gpt-4o-mini`. The API accepts a list of role-based messages as `input`; the response text is available through `response.output_text`.


In [6]:
def return_OAIResponse(user_message, context):
    """Send role-based prompt context and a question to the Responses API."""
    messages = [*context, {"role": "user", "content": f"Question: {user_message}"}]
    response = client.responses.create(
        model=MODEL,
        input=messages,
    )
    return response.output_text


## 1. Zero-shot prompting

With no examples, the model may answer correctly but choose its own format. Inspect the output and note which details are present or missing.


In [7]:
context_user = [
    {"role": "system", "content": "You are an expert in Formula One. Answer factual questions clearly."}
]

In [8]:
# let's see the result with zero-shot:
print(return_OAIResponse("Who won the 2010 F1 Drivers' Championship?", context_user))

Sebastian Vettel won the 2010 F1 Drivers' Championship.


## 2. One-shot prompting

Now include one example as a user/assistant exchange. The example demonstrates the format we want; it does not guarantee that every answer will be correct.


In [9]:
context_user = [
    {"role": "system", "content": "Answer F1 championship questions using exactly these fields: Driver, Team, Points."},
    {"role": "user", "content": "Who won the 2000 F1 Drivers' Championship?"},
    {"role": "assistant", "content": "Driver: Michael Schumacher\nTeam: Ferrari\nPoints: 108"},
]

In [10]:
# let's see the result with one-shot:
print(return_OAIResponse("Who won the 2011 F1 Drivers' Championship?", context_user))

Driver: Sebastian Vettel  
Team: Red Bull Racing  
Points: 392  


## 3. Few-shot prompting

Multiple examples show the format across more than one case. Keep examples accurate and representative of the task.


In [11]:
context_user = [
    {"role": "system", "content": "Answer F1 championship questions using exactly these fields: Driver, Team, Points."},
    {"role": "user", "content": "Who won the 2010 F1 Drivers' Championship?"},
    {"role": "assistant", "content": "Driver: Sebastian Vettel\nTeam: Red Bull Racing\nPoints: 256"},
    {"role": "user", "content": "Who won the 2009 F1 Drivers' Championship?"},
    {"role": "assistant", "content": "Driver: Jenson Button\nTeam: Brawn GP\nPoints: 95"},
]
print(return_OAIResponse("Who won the 2006 F1 Drivers' Championship?", context_user))


Driver: Fernando Alonso  
Team: Renault  
Points: 134  


In [12]:
# let's see the result with few-shot:
print(return_OAIResponse("Who won the 2019 F1 Drivers' Championship?", context_user))

Driver: Lewis Hamilton  
Team: Mercedes  
Points: 413  


## 4. Examples as messages versus instructions

Role-based examples make the demonstration clearer: each example question is a `user` message and its expected answer is an `assistant` message. The system message states the task. You can compare this with a prompt that explains the format directly. Neither approach is always best; test both on the same questions.


In [13]:
context_user = [
    {"role": "system", "content": "You are an expert in Formula One. Answer with exactly these fields, each on its own line: Driver, Team, Points. Use the championship winner's season-end points."}
]
print(return_OAIResponse("Who won the 2019 F1 Drivers' Championship?", context_user))


Driver: Lewis Hamilton  
Team: Mercedes  
Points: 413


## 5. Classification with few-shot examples

Few-shot prompting can also be useful for other types of tasks, including classification. By providing examples of inputs and their expected labels, we can guide a model toward the desired behavior.

In this example, we'll classify product reviews by sentiment into three categories: **Positive**, **Negative**, or **Neutral**. Notice how the examples illustrate both the task and the expected output format.


In [14]:
context_user = [
    {"role": "system", "content": "Classify the product review as Positive, Negative, or Neutral. Return only the label."},
    {"role": "user", "content": "It fulfilled its function perfectly, the price is fair, and I would buy it again."},
    {"role": "assistant", "content": "Positive"},
    {"role": "user", "content": "It did not work well and I would not buy it again. It is too expensive."},
    {"role": "assistant", "content": "Negative"},
    {"role": "user", "content": "I do not know what to say; my son uses it, but he does not love it."},
    {"role": "assistant", "content": "Neutral"},
]
print(return_OAIResponse("I'm not going to return it, but I don't plan to buy it again.", context_user))


Neutral


## Exercise

- Choose a task (for example, extracting fields, classifying feedback, or rewriting text).
    - For example: Extract the product name, price, and category from a product description.
- Create zero-shot, one-shot, and few-shot versions of your prompt.
- Test each version on the same set of at least five inputs.
- Compare correctness, consistency, and adherence to the requested format. Include any failures or hallucinations you observe.
- Write a one-page report summarizing your findings and what you learned.


# Prompt Engineering Exercise: Sentiment Classification

## Objective
The objective of this exercise is to compare zero-shot, one-shot, and few-shot prompting.

## Task
Classify customer product reviews as Positive, Negative, or Neutral.

## Concepts
- Zero-shot prompting: Asking the model to do a task without examples.
- One-shot prompting: Giving the model one example.
- Few-shot prompting: Giving the model several examples.

## Evaluation
I will compare the correctness of the classifications, consistency of the responses, and whether the model follows the requested output format.

In [1]:
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

model_name = "google/flan-t5-base"

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForSeq2SeqLM.from_pretrained(
    model_name
)

model.eval()

print("FLAN-T5-base loaded successfully!")

c:\Users\anita\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
c:\Users\anita\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\anita\.cache\huggingface\hub\models--google--flan-t5-base. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an admi

FLAN-T5-base loaded successfully!


In [2]:
prompt = """
Classify this customer review as Positive, Negative, or Neutral.
Return only one label.

Review: The headphones sound amazing and the battery lasts all day.
Sentiment:
"""

inputs = tokenizer(
    prompt,
    return_tensors="pt",
    truncation=True
)

with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=10
    )

answer = tokenizer.decode(
    outputs[0],
    skip_special_tokens=True
)

print("Model answer:", answer)

Model answer: Positive


In [3]:
reviews = [
    {
        "id": "R1",
        "text": "The headphones sound amazing and the battery lasts all day.",
        "expected": "Positive"
    },
    {
        "id": "R2",
        "text": "The screen stopped working after two days. Very disappointed.",
        "expected": "Negative"
    },
    {
        "id": "R3",
        "text": "The package arrived yesterday. I have not tried the product yet.",
        "expected": "Neutral"
    },
    {
        "id": "R4",
        "text": "Good sound quality, but the battery drains too quickly.",
        "expected": "Neutral"
    },
    {
        "id": "R5",
        "text": "It is okay, nothing special, but it does what it needs to do.",
        "expected": "Neutral"
    }
]

print("Total reviews:", len(reviews))

Total reviews: 5


In [4]:
# ZERO-SHOT: No examples
zero_shot_prompt = """
Classify the customer review as Positive, Negative, or Neutral.

Positive means the overall opinion is favorable.
Negative means the overall opinion is unfavorable.
Neutral means the opinion is mixed, unclear, or no opinion is expressed.

If a review contains both praise and criticism, choose Neutral.

Return only one label: Positive, Negative, or Neutral.

Review: {review}
Sentiment:
"""


# ONE-SHOT: One example
one_shot_prompt = """
Classify the customer review as Positive, Negative, or Neutral.

Positive means favorable.
Negative means unfavorable.
Neutral means mixed, unclear, or no opinion is expressed.

If a review contains both praise and criticism, choose Neutral.

Example:
Review: The camera takes beautiful pictures.
Sentiment: Positive

Return only one label: Positive, Negative, or Neutral.

Review: {review}
Sentiment:
"""


# FEW-SHOT: Three examples
few_shot_prompt = """
Classify the customer review as Positive, Negative, or Neutral.

Positive means favorable.
Negative means unfavorable.
Neutral means mixed, unclear, or no opinion is expressed.

If a review contains both praise and criticism, choose Neutral.

Example 1:
Review: The camera takes beautiful pictures.
Sentiment: Positive

Example 2:
Review: The charger broke after one use.
Sentiment: Negative

Example 3:
Review: The design is nice, but the device overheats.
Sentiment: Neutral

Return only one label: Positive, Negative, or Neutral.

Review: {review}
Sentiment:
"""

print("All three prompts are ready!")

All three prompts are ready!


In [5]:
import pandas as pd
import torch

prompts = {
    "Zero-shot": zero_shot_prompt,
    "One-shot": one_shot_prompt,
    "Few-shot": few_shot_prompt
}

results = []

for method, prompt_template in prompts.items():

    for item in reviews:

        # Insert the review into the prompt
        prompt = prompt_template.format(
            review=item["text"]
        )

        # Convert the prompt into model input
        inputs = tokenizer(
            prompt,
            return_tensors="pt",
            truncation=True
        )

        # Generate the model's answer
        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=10,
                do_sample=False
            )

        # Convert the output into readable text
        answer = tokenizer.decode(
            outputs[0],
            skip_special_tokens=True
        ).strip()

        # Extract the sentiment label
        normalized = answer.lower().strip(" .!,:;")

        if normalized == "positive":
            predicted = "Positive"
        elif normalized == "negative":
            predicted = "Negative"
        elif normalized == "neutral":
            predicted = "Neutral"
        else:
            predicted = "Unclear"

        # Save the result
        results.append({
            "Method": method,
            "Review ID": item["id"],
            "Review": item["text"],
            "Expected": item["expected"],
            "Model Output": answer,
            "Predicted": predicted,
            "Correct": predicted == item["expected"]
        })

results_df = pd.DataFrame(results)

display(results_df)

,Method,Review ID,Review,Expected,Model Output,Predicted,Correct
0,Zero-shot,R1,The headphones sound amazing and the battery l...,Positive,Positive,Positive,True
1,Zero-shot,R2,The screen stopped working after two days. Ver...,Negative,Negative,Negative,True
2,Zero-shot,R3,The package arrived yesterday. I have not trie...,Neutral,Neutral,Neutral,True
3,Zero-shot,R4,"Good sound quality, but the battery drains too...",Neutral,Positive,Positive,False
4,Zero-shot,R5,"It is okay, nothing special, but it does what ...",Neutral,Positive,Positive,False
5,One-shot,R1,The headphones sound amazing and the battery l...,Positive,Positive,Positive,True
6,One-shot,R2,The screen stopped working after two days. Ver...,Negative,Negative,Negative,True
7,One-shot,R3,The package arrived yesterday. I have not trie...,Neutral,Neutral,Neutral,True
8,One-shot,R4,"Good sound quality, but the battery drains too...",Neutral,Neutral,Neutral,True
9,One-shot,R5,"It is okay, nothing special, but it does what ...",Neutral,Positive,Positive,False


In [6]:
summary = results_df.groupby("Method").agg(
    Correct_Answers=("Correct", "sum"),
    Total_Reviews=("Correct", "count")
)

summary["Accuracy (%)"] = (
    summary["Correct_Answers"]
    / summary["Total_Reviews"]
    * 100
).round(1)

display(summary)

,Correct_Answers,Total_Reviews,Accuracy (%)
Method,,,
Few-shot,3,5,60.0
One-shot,4,5,80.0
Zero-shot,3,5,60.0


In [7]:
mistakes = results_df[
    results_df["Correct"] == False
]

if mistakes.empty:
    print("All classifications matched the expected labels.")
else:
    print("Incorrect or unclear classifications:")
    display(
        mistakes[
            [
                "Method",
                "Review ID",
                "Expected",
                "Model Output",
                "Predicted"
            ]
        ]
    )

Incorrect or unclear classifications:


,Method,Review ID,Expected,Model Output,Predicted
3,Zero-shot,R4,Neutral,Positive,Positive
4,Zero-shot,R5,Neutral,Positive,Positive
9,One-shot,R5,Neutral,Positive,Positive
13,Few-shot,R4,Neutral,Positive,Positive
14,Few-shot,R5,Neutral,Positive,Positive


In [8]:
# Check whether each method returned all five reviews
print("Number of results per method:")
display(results_df.groupby("Method").size())

# Check for outputs that are not exactly one of the labels
print("Outputs needing format review:")
display(
    results_df[
        results_df["Predicted"] == "Unclear"
    ][["Method", "Review ID", "Model Output"]]
)

Number of results per method:


Method
Few-shot     5
One-shot     5
Zero-shot    5
dtype: int64

Outputs needing format review:


,Method,Review ID,Model Output


# Report: Comparison of Zero-Shot, One-Shot, and Few-Shot Prompting

## Objective

The objective of this experiment was to compare three prompting techniques for customer review sentiment classification. The task was to classify five product reviews as Positive, Negative, or Neutral using the FLAN-T5-base language model.

## Method

I tested three prompts on the same five reviews. The zero-shot prompt contained instructions without examples, the one-shot prompt contained one labeled example, and the few-shot prompt contained three labeled examples. I compared the results using accuracy, consistency, and adherence to the requested output format.

## Results

The zero-shot method achieved ___% accuracy, the one-shot method achieved ___%, and the few-shot method achieved ___%.

The best-performing method was ___ based on the measured accuracy. I observed ___ incorrect or unclear classifications across the experiments.

One interesting result was the classification of the mixed review about sound quality and battery life. The expected label was Neutral because the review contained both praise and criticism.

## Discussion

Zero-shot prompting relies on instructions alone. One-shot prompting provides an example to demonstrate the expected behavior, while few-shot prompting provides several examples to clarify the classification rules.

The experiment showed that ___. The model [did / did not] always follow the requested output format. An observed failure was ___.

## Conclusion

I learned that prompts can guide the behavior of a language model and that examples may help clarify a task. However, providing more examples does not guarantee better performance. Testing the same inputs and comparing actual results is necessary to determine which prompting technique works best for a particular task.

## Exercise

Your task is to extract structured information from product descriptions.

Use the following product description as your starting example:

> The SoundWave Pro wireless headphones feature noise cancellation, Bluetooth 5.3 connectivity, and up to 30 hours of battery life. Available in black, they cost €89.99 and belong to the Electronics category.

1. Create zero-shot, one-shot, and few-shot prompts to extract the following fields:
   - **Product name**
   - **Price**
   - **Category**

2. Generate three additional product descriptions (using code or an LLM) and test all three prompts on the same descriptions.

3. Compare the results in terms of accuracy, consistency, and adherence to the expected output format. Include any errors or hallucinations you observe.

4. Write a brief, simple report summarizing your findings and what you learned about the impact of examples on model performance.

## Notes

- Model output can vary and may contain factual errors; verify factual answers independently.
- Remember to keep API keys private. API usage may incur charges.
